# DRD2 Hi — Model complexity analysis

This notebook analyses whether the internal validation protocol used for hyperparameter selection changes the complexity of the selected models.

The comparison is between:

- OOD holdout inner validation
- Random shuffle inner validation

The main question is:

**Does random shuffle select models that are more complex and less calibrated for final OOD test generalization?**

The analysis uses the saved per-fold artifacts:

- `params_fold_i.json`
- `complexity_fold_i.json`

In [3]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [4]:
PROJECT_ROOT = Path("../..").resolve()

RAW_RESULTS_DIR = PROJECT_ROOT / "results" / "hi" / "drd2"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / "hi"
    / "drd2"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw results:", RAW_RESULTS_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: /home/f.capria/drug-discovery-lohi
Raw results: /home/f.capria/drug-discovery-lohi/results/hi/drd2
Output dir: /home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/drd2


## Experiment registry

The registry defines which result folders correspond to each combination of:

- model
- fingerprint
- protocol

In [5]:
EXPERIMENTS = [
    # Decision Tree
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_drd2_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_drd2_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_drd2_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_drd2_hi_random_shuffle_maccs",
    },

    # Logistic Regression
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_drd2_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_drd2_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_drd2_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_drd2_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_drd2_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_drd2_hi_random_shuffle_rdkit_desc",
    },

    # Linear SVM
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_drd2_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_drd2_hi_random_shuffle_ecfp4",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_drd2_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "SVM linear",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_drd2_hi_random_shuffle_maccs",
    },
]

In [6]:
def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def safe_get(dct, key, default=np.nan):
    return dct.get(key, default)

In [7]:
def load_complexity_rows(experiments, raw_results_dir):
    rows = []

    for exp in experiments:
        result_dir = raw_results_dir / exp["result_dir"]

        if not result_dir.exists():
            print(f"Missing directory: {result_dir}")
            continue

        for fold in [1, 2, 3]:
            params_path = result_dir / f"params_fold_{fold}.json"
            complexity_path = result_dir / f"complexity_fold_{fold}.json"

            if not params_path.exists():
                print(f"Missing params file: {params_path}")
                continue

            if not complexity_path.exists():
                print(f"Missing complexity file: {complexity_path}")
                continue

            params = read_json(params_path)
            complexity = read_json(complexity_path)

            train_metrics = params.get("train_metrics", {})
            test_metrics = params.get("test_metrics", {})

            inner_pr_auc = params.get("inner_selection_score", np.nan)
            inner_train_pr_auc = params.get("inner_train_score", np.nan)
            train_pr_auc = train_metrics.get("pr_auc", np.nan)
            test_pr_auc = test_metrics.get("pr_auc", np.nan)

            row = {
                "model": exp["model"],
                "model_short": exp["model_short"],
                "fingerprint": exp["fingerprint"],
                "protocol": exp["protocol"],
                "result_dir": exp["result_dir"],
                "fold": fold,

                # Scores
                "inner_pr_auc": inner_pr_auc,
                "inner_train_pr_auc": inner_train_pr_auc,
                "train_pr_auc": train_pr_auc,
                "test_pr_auc": test_pr_auc,

                # Gaps
                "train_inner_gap": train_pr_auc - inner_pr_auc,
                "inner_test_gap": inner_pr_auc - test_pr_auc,
                "train_test_gap": train_pr_auc - test_pr_auc,
            }

            # Add all complexity fields
            for key, value in complexity.items():
                row[key] = value

            rows.append(row)

    return pd.DataFrame(rows)

In [9]:
complexity_all = load_complexity_rows(EXPERIMENTS, RAW_RESULTS_DIR)

complexity_all = complexity_all.sort_values(
    ["model_short", "fingerprint", "protocol", "fold"]
).reset_index(drop=True)

complexity_all.head(3)

,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,...,l1_norm,l2_norm,approx_margin,intercept,C,penalty,l1_ratio,dual,n_support_per_class,n_support_total
0,Decision Tree,DT,ECFP4,OOD holdout,dt_drd2_hi_inner_ood_holdout_ecfp4,1,0.855177,0.917171,0.9080,0.6979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Decision Tree,DT,ECFP4,OOD holdout,dt_drd2_hi_inner_ood_holdout_ecfp4,2,0.699841,0.873181,0.8189,0.7698,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Decision Tree,DT,ECFP4,OOD holdout,dt_drd2_hi_inner_ood_holdout_ecfp4,3,0.737516,0.905641,0.8232,0.6815,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print("Rows loaded:", len(complexity_all))
print("Expected rows:", len(EXPERIMENTS) * 3)

complexity_all[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
        "model_class",
    ]
]

Rows loaded: 42
Expected rows: 42


,model,fingerprint,protocol,fold,inner_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,model_class
0,Decision Tree,ECFP4,OOD holdout,1,0.855177,0.9080,0.6979,0.157277,0.2101,DecisionTreeClassifier
1,Decision Tree,ECFP4,OOD holdout,2,0.699841,0.8189,0.7698,-0.069959,0.0491,DecisionTreeClassifier
2,Decision Tree,ECFP4,OOD holdout,3,0.737516,0.8232,0.6815,0.056016,0.1417,DecisionTreeClassifier
3,Decision Tree,ECFP4,Random shuffle,1,0.911535,0.9522,0.6237,0.287835,0.3285,DecisionTreeClassifier
4,Decision Tree,ECFP4,Random shuffle,2,0.844398,0.8757,0.7540,0.090398,0.1217,DecisionTreeClassifier
5,Decision Tree,ECFP4,Random shuffle,3,0.909690,0.9573,0.6544,0.255290,0.3029,DecisionTreeClassifier
6,Decision Tree,MACCS,OOD holdout,1,0.842697,0.9679,0.6262,0.216497,0.3417,DecisionTreeClassifier
7,Decision Tree,MACCS,OOD holdout,2,0.741046,0.8447,0.7569,-0.015854,0.0878,DecisionTreeClassifier
8,Decision Tree,MACCS,OOD holdout,3,0.721427,0.7870,0.6575,0.063927,0.1295,DecisionTreeClassifier
9,Decision Tree,MACCS,Random shuffle,1,0.915752,0.9554,0.5913,0.324452,0.3641,DecisionTreeClassifier


## Logistic Regression complexity table

For Logistic Regression complexity is described by:

- selected `C` - larger --> weaker regularization
- selected `l1_ratio`
- number of non-zero coefficients
- sparsity
- L1 norm of the coefficient vector
- L2 norm of the coefficient vector - 

In [11]:
lr_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "l1_ratio",
    "class_weight",
    "n_nonzero_coefficients",
    "sparsity",
    "l1_norm",
    "l2_norm",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

lr_table = (
    complexity_all[complexity_all["model_short"] == "LR"]
    [lr_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

lr_table

,model,fingerprint,protocol,fold,C,l1_ratio,class_weight,n_nonzero_coefficients,sparsity,l1_norm,l2_norm,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Logistic Regression,ECFP4,OOD holdout,1,0.10,0.25,balanced,376.0,0.632812,60.246652,4.033987,0.860715,0.6795,0.181215,0.2879
1,Logistic Regression,ECFP4,OOD holdout,2,0.10,0.00,balanced,1022.0,0.001953,135.150989,5.425810,0.667518,0.8695,-0.201982,0.0911
2,Logistic Regression,ECFP4,OOD holdout,3,0.05,1.00,balanced,34.0,0.966797,7.924230,1.759531,0.726085,0.7226,0.003485,0.1816
3,Logistic Regression,ECFP4,Random shuffle,1,1.00,0.25,balanced,792.0,0.226562,342.904666,15.375631,0.936648,0.6486,0.288048,0.3446
4,Logistic Regression,ECFP4,Random shuffle,2,0.50,0.25,balanced,714.0,0.302734,223.949768,10.657827,0.880490,0.8660,0.014490,0.1113
5,Logistic Regression,ECFP4,Random shuffle,3,0.10,0.00,balanced,1020.0,0.003906,127.358589,5.280609,0.929450,0.7884,0.141050,0.1819
6,Logistic Regression,MACCS,OOD holdout,1,1.00,1.00,NaN,102.0,0.389222,56.381426,7.406115,0.804256,0.6403,0.163956,0.2791
7,Logistic Regression,MACCS,OOD holdout,2,10.00,1.00,NaN,140.0,0.161677,150.403058,21.548625,0.690991,0.8075,-0.116509,0.0844
8,Logistic Regression,MACCS,OOD holdout,3,10.00,1.00,balanced,137.0,0.179641,147.209510,20.458413,0.717985,0.7904,-0.072415,0.1222
9,Logistic Regression,MACCS,Random shuffle,1,10.00,0.50,balanced,144.0,0.137725,124.060243,14.785380,0.882713,0.6598,0.222913,0.2706


## Linear SVM complexity table

For linear SVM the indicators are:

- selected `C`
- L2 norm of the weight vector
- approximate margin

In [12]:
svm_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "C",
    "class_weight",
    "l2_norm",
    "approx_margin",
    "n_nonzero_coefficients",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

svm_table = (
    complexity_all[complexity_all["model_short"] == "SVM"]
    [svm_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

svm_table

,model,fingerprint,protocol,fold,C,class_weight,l2_norm,approx_margin,n_nonzero_coefficients,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,SVM linear,ECFP4,OOD holdout,1,100.000,NaN,63.729514,0.015691,990.0,0.853122,0.6434,0.209722,0.3537
1,SVM linear,ECFP4,OOD holdout,2,0.010,balanced,2.161980,0.462539,1017.0,0.673759,0.8613,-0.187541,0.0596
2,SVM linear,ECFP4,OOD holdout,3,0.001,NaN,0.334643,2.988262,992.0,0.686163,0.7550,-0.068837,0.1564
3,SVM linear,ECFP4,Random shuffle,1,0.250,balanced,9.330527,0.107175,999.0,0.933897,0.6587,0.275197,0.3291
4,SVM linear,ECFP4,Random shuffle,2,0.100,balanced,6.346793,0.157560,1011.0,0.872431,0.8714,0.001031,0.0943
5,SVM linear,ECFP4,Random shuffle,3,0.100,balanced,6.206736,0.161115,1004.0,0.921689,0.7873,0.134389,0.1868
6,SVM linear,MACCS,OOD holdout,1,1.000,NaN,5.650166,0.176986,147.0,0.809885,0.6489,0.160985,0.2564
7,SVM linear,MACCS,OOD holdout,2,25.000,balanced,15.712576,0.063643,148.0,0.713134,0.7832,-0.070066,0.1010
8,SVM linear,MACCS,OOD holdout,3,50.000,balanced,15.964132,0.062640,143.0,0.714653,0.7860,-0.071347,0.1203
9,SVM linear,MACCS,Random shuffle,1,25.000,balanced,12.681475,0.078855,146.0,0.878878,0.6559,0.222978,0.2685


## Decision Tree complexity table

For Decision Trees:

- selected `ccp_alpha`
- selected `max_depth`
- effective tree depth
- number of nodes
- number of leaves
- number of features used in the tree
- average minimum depth of the used features

The number of nodes and leaves gives a direct indication of how large the fitted tree is.

In [13]:
dt_cols = [
    "model",
    "fingerprint",
    "protocol",
    "fold",
    "ccp_alpha",
    "max_depth",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "used_feature_fraction",
    "feature_min_depth_mean",
    "feature_min_depth_std",
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

dt_table = (
    complexity_all[complexity_all["model_short"] == "DT"]
    [dt_cols]
    .sort_values(["fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

dt_table

,model,fingerprint,protocol,fold,ccp_alpha,max_depth,effective_depth,n_nodes,n_leaves,n_features_used,used_feature_fraction,feature_min_depth_mean,feature_min_depth_std,inner_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.0010,20.0,20.0,101.0,51.0,48.0,0.046875,8.395833,4.902124,0.855177,0.6979,0.157277,0.2101
1,Decision Tree,ECFP4,OOD holdout,2,0.0010,15.0,15.0,75.0,38.0,37.0,0.036133,6.594595,3.962387,0.699841,0.7698,-0.069959,0.0491
2,Decision Tree,ECFP4,OOD holdout,3,0.0001,7.0,7.0,75.0,38.0,35.0,0.034180,4.114286,1.634825,0.737516,0.6815,0.056016,0.1417
3,Decision Tree,ECFP4,Random shuffle,1,0.0000,20.0,20.0,303.0,152.0,127.0,0.124023,9.385827,4.915102,0.911535,0.6237,0.287835,0.3285
4,Decision Tree,ECFP4,Random shuffle,2,0.0001,15.0,15.0,173.0,87.0,78.0,0.076172,7.448718,3.532441,0.844398,0.7540,0.090398,0.1217
5,Decision Tree,ECFP4,Random shuffle,3,0.0000,20.0,20.0,215.0,108.0,86.0,0.083984,7.709302,4.374520,0.909690,0.6544,0.255290,0.3029
6,Decision Tree,MACCS,OOD holdout,1,0.0001,15.0,15.0,597.0,299.0,100.0,0.598802,6.770000,2.697610,0.842697,0.6262,0.216497,0.3417
7,Decision Tree,MACCS,OOD holdout,2,0.0001,7.0,7.0,101.0,51.0,37.0,0.221557,4.081081,1.633739,0.741046,0.7569,-0.015854,0.0878
8,Decision Tree,MACCS,OOD holdout,3,0.0100,7.0,3.0,11.0,6.0,5.0,0.029940,1.200000,0.748331,0.721427,0.6575,0.063927,0.1295
9,Decision Tree,MACCS,Random shuffle,1,0.0000,20.0,18.0,389.0,195.0,94.0,0.562874,7.265957,3.166230,0.915752,0.5913,0.324452,0.3641


## Gap analysis table

This table collects the three main performance levels:

- train PR-AUC
- inner validation PR-AUC
- final OOD test PR-AUC

and the corresponding gaps:

$$\text{train-inner gap} = \text{train PR-AUC} - \text{inner PR-AUC}$$

$$\text{inner-test gap} = \text{inner PR-AUC} - \text{test PR-AUC}$$

$$\text{train-test gap} = \text{train PR-AUC} - \text{test PR-AUC}$$


This table connects model selection, overfitting and OOD generalization.

In [14]:
gap_cols = [
    "model",
    "model_short",
    "fingerprint",
    "protocol",
    "fold",
    "train_pr_auc",
    "inner_pr_auc",
    "inner_train_pr_auc",
    "test_pr_auc",
    "train_inner_gap",
    "inner_test_gap",
    "train_test_gap",
]

gap_analysis = (
    complexity_all[gap_cols]
    .sort_values(["model_short", "fingerprint", "protocol", "fold"])
    .reset_index(drop=True)
)

gap_analysis

,model,model_short,fingerprint,protocol,fold,train_pr_auc,inner_pr_auc,inner_train_pr_auc,test_pr_auc,train_inner_gap,inner_test_gap,train_test_gap
0,Decision Tree,DT,ECFP4,OOD holdout,1,0.9080,0.855177,0.917171,0.6979,0.052823,0.157277,0.2101
1,Decision Tree,DT,ECFP4,OOD holdout,2,0.8189,0.699841,0.873181,0.7698,0.119059,-0.069959,0.0491
2,Decision Tree,DT,ECFP4,OOD holdout,3,0.8232,0.737516,0.905641,0.6815,0.085684,0.056016,0.1417
3,Decision Tree,DT,ECFP4,Random shuffle,1,0.9522,0.911535,0.960189,0.6237,0.040665,0.287835,0.3285
4,Decision Tree,DT,ECFP4,Random shuffle,2,0.8757,0.844398,0.904329,0.7540,0.031302,0.090398,0.1217
5,Decision Tree,DT,ECFP4,Random shuffle,3,0.9573,0.909690,0.959572,0.6544,0.047610,0.255290,0.3029
6,Decision Tree,DT,MACCS,OOD holdout,1,0.9679,0.842697,0.936617,0.6262,0.125203,0.216497,0.3417
7,Decision Tree,DT,MACCS,OOD holdout,2,0.8447,0.741046,0.861852,0.7569,0.103654,-0.015854,0.0878
8,Decision Tree,DT,MACCS,OOD holdout,3,0.7870,0.721427,0.845624,0.6575,0.065573,0.063927,0.1295
9,Decision Tree,DT,MACCS,Random shuffle,1,0.9554,0.915752,0.956260,0.5913,0.039648,0.324452,0.3641


## Aggregated complexity summary


In [15]:
summary_metrics = [
    "inner_pr_auc",
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
    "l2_norm",
    "n_nonzero_coefficients",
    "approx_margin",
    "effective_depth",
    "n_nodes",
    "n_leaves",
    "n_features_used",
    "feature_min_depth_mean",
]

available_summary_metrics = [
    col for col in summary_metrics
    if col in complexity_all.columns
]

complexity_summary = (
    complexity_all
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    [available_summary_metrics]
    .agg(["mean", "std"])
)

complexity_summary.columns = [
    "_".join([c for c in col if c])
    for col in complexity_summary.columns
]

complexity_summary = complexity_summary.reset_index()

complexity_summary

,index,model,model_short,fingerprint,protocol,inner_pr_auc_mean,inner_pr_auc_std,test_pr_auc_mean,test_pr_auc_std,inner_test_gap_mean,...,effective_depth_mean,effective_depth_std,n_nodes_mean,n_nodes_std,n_leaves_mean,n_leaves_std,n_features_used_mean,n_features_used_std,feature_min_depth_mean_mean,feature_min_depth_mean_std
0,0,Decision Tree,DT,ECFP4,OOD holdout,0.764178,0.081028,0.716400,0.046967,0.047778,...,14.000000,6.557439,83.666667,15.011107,42.333333,7.505553,40.000000,7.000000,6.368238,2.149730
1,1,Decision Tree,DT,ECFP4,Random shuffle,0.888541,0.038240,0.677367,0.068118,0.211174,...,18.333333,2.886751,230.333333,66.342545,115.666667,33.171273,97.000000,26.286879,8.181282,1.051271
2,2,Decision Tree,DT,MACCS,OOD holdout,0.768390,0.065095,0.680200,0.068243,0.088190,...,8.333333,6.110101,236.333333,315.571439,118.666667,157.785720,47.333333,48.335632,4.017027,2.785552
3,3,Decision Tree,DT,MACCS,Random shuffle,0.888069,0.034918,0.673933,0.082550,0.214136,...,17.666667,2.516611,435.666667,80.829038,218.333333,40.414519,95.666667,5.686241,6.894693,0.338131
4,4,Logistic Regression,LR,ECFP4,OOD holdout,0.751440,0.099063,0.757200,0.099614,-0.005760,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,Logistic Regression,LR,ECFP4,Random shuffle,0.915529,0.030558,0.767667,0.110173,0.147863,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,Logistic Regression,LR,MACCS,OOD holdout,0.737744,0.059161,0.746067,0.091995,-0.008323,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,Logistic Regression,LR,MACCS,Random shuffle,0.870747,0.013825,0.751633,0.079851,0.119114,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,Logistic Regression,LR,RDKit desc,OOD holdout,0.764256,0.076487,0.788267,0.080877,-0.024010,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,Logistic Regression,LR,RDKit desc,Random shuffle,0.877498,0.034276,0.771300,0.089346,0.106198,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
complexity_all.to_csv(OUTPUT_DIR / "complexity_all.csv", index=False)
lr_table.to_csv(OUTPUT_DIR / "complexity_lr.csv", index=False)
svm_table.to_csv(OUTPUT_DIR / "complexity_svm.csv", index=False)
dt_table.to_csv(OUTPUT_DIR / "complexity_dt.csv", index=False)
gap_analysis.to_csv(OUTPUT_DIR / "complexity_gap_analysis.csv", index=False)
complexity_summary.to_csv(OUTPUT_DIR / "complexity_summary.csv", index=False)

print("Saved complexity tables in:")
print(OUTPUT_DIR)

Saved complexity tables in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/drd2
